# Brick 1 — Pure ecological model

Interactive exploration of the Lotka-Volterra dynamics of the `bilevel-fishery` framework.

**Objectives**:
1. See trajectories with no fishing pressure
2. See the effect of **moderate** fishing (system finds a new equilibrium)
3. See the effect of **excessive** fishing (collapse → `EcologyInstabilityError`)
4. Compare Euler vs RK45 for different `dt`

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from bilevel_fishery.ecology import (
    EcologicalState,
    EcologyInstabilityError,
    EcologyParams,
    step,
)

plt.rcParams["figure.figsize"] = (10, 5)

## 1. Free trajectory (no fishing)

At default parameters, the equilibrium is at
$(F^*, A^*) = (\alpha/\beta,\ \gamma/\delta) = (10,\ 20)$.
We start slightly off-equilibrium to see the oscillations.

In [ ]:
params = EcologyParams(dt=0.05, integrator="rk45")

state = EcologicalState(fish=15.0, algae=15.0)
fish_traj = [state.fish]
algae_traj = [state.algae]

for _ in range(400):
    state = step(state, params, harvest=0.0)
    fish_traj.append(state.fish)
    algae_traj.append(state.algae)

t = np.arange(len(fish_traj)) * params.dt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.5))
ax1.plot(t, fish_traj, label="fish (predator)")
ax1.plot(t, algae_traj, label="algae (prey)")
ax1.axhline(
    params.alpha / params.beta, color="C0", ls=":", label=r"$F^* = \alpha/\beta$"
)
ax1.axhline(
    params.gamma / params.delta, color="C1", ls=":", label=r"$A^* = \gamma/\delta$"
)
ax1.set_xlabel("time")
ax1.set_ylabel("biomass")
ax1.set_title("Time series (no harvest)")
ax1.legend()

ax2.plot(algae_traj, fish_traj, lw=0.8)
ax2.plot(
    [params.gamma / params.delta],
    [params.alpha / params.beta],
    "r*",
    ms=12,
    label="equilibrium",
)
ax2.set_xlabel("algae")
ax2.set_ylabel("fish")
ax2.set_title("Phase portrait (closed orbit)")
ax2.legend()
plt.tight_layout()
plt.show()

## 2. Moderate fishing: the system finds a new equilibrium

With fishing pressure **below the endogenous production**, the fish stock
decreases then oscillates around a lower equilibrium. Algae grow slightly
to compensate.

In [ ]:
params = EcologyParams(dt=0.05, integrator="rk45")
harvests = [0.0, 0.2, 0.4]

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5), sharey=True)
for ax, harvest in zip(axes, harvests, strict=True):
    state = EcologicalState(fish=15.0, algae=15.0)
    fish_traj = [state.fish]
    algae_traj = [state.algae]
    for _ in range(400):
        state = step(state, params, harvest=harvest)
        fish_traj.append(state.fish)
        algae_traj.append(state.algae)
    t = np.arange(len(fish_traj)) * params.dt
    ax.plot(t, fish_traj, label="fish")
    ax.plot(t, algae_traj, label="algae")
    ax.set_xlabel("time")
    ax.set_title(f"harvest = {harvest}")
    ax.legend()
axes[0].set_ylabel("biomass")
plt.tight_layout()
plt.show()

## 3. Excessive fishing: collapse and `EcologyInstabilityError`

When fishing pressure exceeds endogenous production, the stock collapses
towards 0. Numerically, the RK45 integrator eventually produces a
`fish < 0` (non-physical territory) and our `step()` **refuses to continue**
by raising an `EcologyInstabilityError`.

This is a **deliberate fail-loud**: we prefer an explicit crash over an
incorrect behaviour (such as the silent clamp to 0 in the master codebase).

In [ ]:
params = EcologyParams(dt=0.05, integrator="rk45")
state = EcologicalState(fish=15.0, algae=15.0)
fish_traj = [state.fish]
algae_traj = [state.algae]

crashed_at: int | None = None
max_steps = 500
for i in range(max_steps):
    try:
        state = step(state, params, harvest=2.0)
    except EcologyInstabilityError as err:
        crashed_at = i
        print(f"Crashed at step {i} (t={i * params.dt:.2f}): {err}")
        break
    fish_traj.append(state.fish)
    algae_traj.append(state.algae)

t = np.arange(len(fish_traj)) * params.dt
fig, ax = plt.subplots()
ax.plot(t, fish_traj, label="fish")
ax.plot(t, algae_traj, label="algae")
ax.set_xlabel("time")
ax.set_ylabel("biomass")
if crashed_at is not None:
    ax.axvline(
        crashed_at * params.dt,
        color="red",
        ls="--",
        label=f"crash @ t={crashed_at * params.dt:.2f}",
    )
ax.set_title("Over-harvest (H=2.0): collapse")
ax.legend()
plt.show()

## 4. Euler vs RK45: why the solver choice matters

For a moderate `dt`, Euler accumulates amplitude error (oscillations grow
artificially). RK45 stays faithful to the true dynamics.

In [ ]:
dts = [0.05, 0.2]


def integrate(params: EcologyParams, n_steps: int) -> list[float]:
    """Return fish trajectory; stop early if EcologyInstabilityError fires."""
    state = EcologicalState(fish=15.0, algae=15.0)
    traj = [state.fish]
    for _ in range(n_steps):
        try:
            state = step(state, params, harvest=0.0)
        except EcologyInstabilityError:
            break
        traj.append(state.fish)
    return traj


fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for ax, dt in zip(axes, dts, strict=True):
    p_euler = EcologyParams(dt=dt, integrator="euler")
    p_rk45 = EcologyParams(dt=dt, integrator="rk45")
    n_steps = int(20 / dt)
    fish_e = integrate(p_euler, n_steps)
    fish_r = integrate(p_rk45, n_steps)
    ax.plot(np.arange(len(fish_e)) * dt, fish_e, label="Euler", lw=1.5)
    ax.plot(np.arange(len(fish_r)) * dt, fish_r, label="RK45", lw=1.5)
    ax.set_title(f"fish biomass, dt = {dt}")
    ax.set_xlabel("time")
    ax.legend()
axes[0].set_ylabel("fish")
plt.tight_layout()
plt.show()

## 5. Takeaways

- Without fishing, the Lotka-Volterra model **oscillates** around a stable
  centre.
- A **moderate pressure** lowers the equilibrium: the system stays viable.
- An **excessive pressure** leads to collapse. Our `step()` raises
  `EcologyInstabilityError` rather than masking the problem with a silent
  clamp (master codebase anti-pattern).
- **Euler diverges** as `dt` grows; **RK45 stays faithful** because it
  subdivides the step internally.
- This is the module that will be **wrapped as a Gymnasium environment**
  in **Brick 2**.